In [2]:
!pip install psycopg2-binary boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 4.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 4.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 4.2 MB/s eta 0:00:00


In [3]:
import pandas as pd
from datetime import datetime
import os, psycopg2, boto3
from io import StringIO
import numpy as np

POSTGRES_HOST = os.getenv("POSTGRES_HOST", "host.docker.internal")
POSTGRES_PORT = int(os.getenv("POSTGRES_PORT", "5432"))
POSTGRES_DB = os.getenv("POSTGRES_DB", "seminar_work")
POSTGRES_USER = os.getenv("POSTGRES_USER", "postgres")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD", "postgres")

MINIO_HOST = os.getenv("MINIO_HOST", "host.docker.internal")
MINIO_PORT = int(os.getenv("MINIO_PORT", "9000"))
MINIO_USER = os.getenv("MINIO_ROOT_USER", "admin")
MINIO_PASS = os.getenv("MINIO_ROOT_PASSWORD", "password")
MINIO_BUCKET = os.getenv("MINIO_BUCKET", "seminar")

production = os.getenv("PG_SOURCE_TABLE1", "production")
well_telemetry = os.getenv("PG_SOURCE_TABLE2", "well_telemetry")  
wells = os.getenv("PG_SOURCE_TABLE3", "wells")
S3_PREFIX = os.getenv("S3_EXPORT_PREFIX", "exports/")

def export_pg_to_minio():
    conn = psycopg2.connect(
        host=POSTGRES_HOST, port=POSTGRES_PORT, dbname=POSTGRES_DB,
        user=POSTGRES_USER, password=POSTGRES_PASSWORD
    )
    
    df_production = pd.read_sql_query(f"SELECT prod_id, well_id, date, oil_ton, gas_m3, water_m3, energy_kwh, downtime_hours, temperature, pressure FROM {production}", conn)
    df_telemetry = pd.read_sql_query(f"SELECT record_id, well_id, timestamp, pump_speed_rpm, pump_current, pressure_in, pressure_out, temperature, vibration, oil_flow_rate FROM {well_telemetry}", conn)
    df_wells = pd.read_sql_query(f"SELECT well_id, name, field_name, region, start_date, operator, status FROM {wells}", conn)

    df_production['temperature'] = df_production['temperature'].fillna(df_production['temperature'].median())
    df_production['pressure'] = df_production['pressure'].fillna(df_production['pressure'].median())

    df_production = df_production[df_production['oil_ton'] <= 1000]
    df_production = df_production[(df_production['pressure'] >= 0) & (df_production['pressure'] <= 500)]
    df_production = df_production[(df_production['temperature'] >= -30) & (df_production['temperature'] <= 200)]
    df_production = df_production[(df_production['downtime_hours'] >= 0) & (df_production['downtime_hours'] <= 24)]

    df_production['avg_pressure'] = np.mean(df_production['pressure'])
    df_production['avg_temperature'] = np.mean(df_production['temperature'])
    df_production['downtime_coef'] = df_production['downtime_hours'] / 24

    df_telemetry = df_telemetry[(df_telemetry['pump_speed_rpm'] >= 0) & (df_telemetry['pump_speed_rpm'] <= 3000)]
    df_telemetry = df_telemetry[(df_telemetry['temperature'] >= -30) & (df_telemetry['temperature'] <= 150)]
    df_telemetry = df_telemetry[(df_telemetry['vibration'] >= 0) & (df_telemetry['vibration'] <= 50)]

    df_telemetry['avg_pressure'] = (df_telemetry['pressure_in'] + df_telemetry['pressure_out']) / 2
    df_telemetry['date'] = df_telemetry['timestamp'].dt.date
    
    conn.close()
    
    s3 = boto3.client(
        "s3",
        endpoint_url=f"http://{MINIO_HOST}:{MINIO_PORT}",
        aws_access_key_id=MINIO_USER,
        aws_secret_access_key=MINIO_PASS,
    )
    
    timestamp = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
    
    for name, df in [("production", df_production), ("well_telemetry", df_telemetry), ("wells", df_wells)]:
        csv_buffer = StringIO()
        df.to_csv(csv_buffer, index=False)
        key = f"{S3_PREFIX}{name}_{timestamp}.csv"
        s3.put_object(Bucket=MINIO_BUCKET, Key=key, Body=csv_buffer.getvalue().encode('utf-8'))

In [5]:
export_pg_to_minio()

/tmp/ipykernel_177/511770430.py:30: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_production = pd.read_sql_query(f"SELECT prod_id, well_id, date, oil_ton, gas_m3, water_m3, energy_kwh, downtime_hours, temperature, pressure FROM {production}", conn)
/tmp/ipykernel_177/511770430.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_telemetry = pd.read_sql_query(f"SELECT record_id, well_id, timestamp, pump_speed_rpm, pump_current, pressure_in, pressure_out, temperature, vibration, oil_flow_rate FROM {well_telemetry}", conn)
/tmp/ipykernel_177/511770430.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 

In [6]:
POSTGRES_HOST = os.getenv("POSTGRES_HOST", "host.docker.internal")
POSTGRES_PORT = int(os.getenv("POSTGRES_PORT", "5432"))
POSTGRES_DB = os.getenv("POSTGRES_DB", "seminar_work")
POSTGRES_USER = os.getenv("POSTGRES_USER", "postgres")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD", "postgres")

conn = psycopg2.connect(
        host=POSTGRES_HOST, port=POSTGRES_PORT, dbname=POSTGRES_DB,
        user=POSTGRES_USER, password=POSTGRES_PASSWORD
    )
    
   
df_production = pd.read_sql_query(f"SELECT prod_id, well_id, date, oil_ton, gas_m3, water_m3, energy_kwh, downtime_hours, temperature, pressure FROM {production}", conn)
df_telemetry = pd.read_sql_query(f"SELECT record_id, well_id, timestamp, pump_speed_rpm, pump_current, pressure_in, pressure_out, temperature, vibration, oil_flow_rate FROM {well_telemetry}", conn)
df_wells = pd.read_sql_query(f"SELECT well_id, name, field_name, region, start_date, operator, status FROM {wells}", conn)

    #conn.close()

/tmp/ipykernel_177/917326584.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_production = pd.read_sql_query(f"SELECT prod_id, well_id, date, oil_ton, gas_m3, water_m3, energy_kwh, downtime_hours, temperature, pressure FROM {production}", conn)
/tmp/ipykernel_177/917326584.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_telemetry = pd.read_sql_query(f"SELECT record_id, well_id, timestamp, pump_speed_rpm, pump_current, pressure_in, pressure_out, temperature, vibration, oil_flow_rate FROM {well_telemetry}", conn)
/tmp/ipykernel_177/917326584.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 

In [7]:
import pandas as pd
import numpy as np


df_production['date'] = pd.to_datetime(df_production['date'])

daily_agg = df_production.groupby('date').agg({
    'oil_ton': 'sum',
    'gas_m3': 'sum',
    'water_m3': 'sum',
    'downtime_hours': 'mean'
})


well_agg = df_production.groupby('well_id').agg({
    'oil_ton': ['sum', 'mean'],
    'gas_m3': 'sum',
    'water_m3': 'sum',
    'downtime_hours': 'mean'
})

df_telemetry['date'] = df_telemetry['timestamp']
df_telemetry['date'] = pd.to_datetime(df_telemetry['date'])


daily_tele = df_telemetry.groupby('date').agg({
    'pump_speed_rpm': 'mean',
    'pump_current': 'mean',
    'pressure_in': 'mean',
    'pressure_out': 'mean',
    'temperature': 'mean',
    'vibration': 'mean',
    'oil_flow_rate': 'mean'
})

print(df_production.info())
print('-----------------------------------------------------------')
print(daily_agg.head())
print('-----------------------------------------------------------')
print(well_agg.head())
print('-----------------------------------------------------------')
print(daily_tele.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   prod_id         150 non-null    int64         
 1   well_id         150 non-null    int64         
 2   date            150 non-null    datetime64[ns]
 3   oil_ton         150 non-null    float64       
 4   gas_m3          150 non-null    float64       
 5   water_m3        150 non-null    float64       
 6   energy_kwh      150 non-null    float64       
 7   downtime_hours  150 non-null    float64       
 8   temperature     120 non-null    float64       
 9   pressure        120 non-null    float64       
dtypes: datetime64[ns](1), float64(7), int64(2)
memory usage: 11.8 KB
None
-----------------------------------------------------------
            oil_ton    gas_m3  water_m3  downtime_hours
date                                                   
2025-10-01    717.5  197

# Практическое задание 2

In [14]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sqlalchemy import create_engine
from sklearn.impute import SimpleImputer

engine = create_engine(f'postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}')

df = pd.read_sql("SELECT date, well_id, oil_ton, pressure, temperature, energy_kwh, downtime_hours FROM production", engine)

imputer = SimpleImputer(strategy='mean')
df[['pressure', 'temperature']] = imputer.fit_transform(df[['pressure', 'temperature']])

df['operating_hours'] = 24 - df['downtime_hours'].fillna(0)
features = ['pressure', 'temperature', 'energy_kwh', 'operating_hours']
X = df[features]
y = df['oil_ton']

model = RandomForestRegressor()
model.fit(X, y)
df['predicted'] = model.predict(X)

print(f"MAE: {mean_absolute_error(y, df['predicted']):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y, df['predicted'])):.2f}")

df.to_sql('debit_predictions', engine, if_exists='replace')

MAE: 0.07
RMSE: 0.10


150

# Практическое задание 3

In [27]:
from sklearn.ensemble import IsolationForest, RandomForestClassifier

engine = create_engine(f'postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}')

df = pd.read_sql("SELECT * FROM pump_sensors", engine)
failures = pd.read_sql("SELECT * FROM pump_failures", engine) 

features = ['vibration', 'temperature', 'current', 'rpm', 'pressure']

df['anomaly'] = IsolationForest(contamination=0.1).fit_predict(df[features])
df['anomaly'] = (df['anomaly'] == -1).astype(int)

#-------------------------------------------------------------------------

df['failure_soon'] = 0

for _, failure in failures.iterrows():
    mask = (df['pump_id'] == failure['pump_id']) & \
           (df['timestamp'] < failure['failure_date']) & \
           (df['timestamp'] >= failure['failure_date'] - pd.Timedelta(hours=24))
    df.loc[mask, 'failure_soon'] = 1

#-------------------------------------------------------------------------

model = RandomForestClassifier()
model.fit(df[features], df['failure_soon'])

df['risk_score'] = model.predict_proba(df[features])[:, 1]

#-------------------------------------------------------------------------
df.to_sql('pump_risk', engine, if_exists='replace')

print(f"Всего записей: {len(df)}")
print(f"Аномалий найдено: {df['anomaly'].sum()}")



importance = pd.DataFrame({'признак': features,
                           'важность': model.feature_importances_}).sort_values('важность', ascending=False)


print(importance)

importance.to_sql('factors_risk', engine, if_exists='replace')

Всего записей: 72
Аномалий найдено: 8
       признак  важность
4     pressure  0.290349
1  temperature  0.251886
0    vibration  0.226192
2      current  0.143800
3          rpm  0.087774


5

# Практическое задание 4

In [29]:
engine = create_engine(f'postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}')

df = pd.read_sql("SELECT * FROM deliveries", engine)

df['cost_per_km'] = df['cost_usd'] / df['distance_km']

print("=== Средняя задержка по погоде ===")
print(df.groupby('weather_conditions')['delay_hours'].mean())

print("=== Средняя задержка по водителям ===")
print(df.groupby('driver_id')['delay_hours'].mean().sort_values())

print("=== Корреляция расстояние и задежка ===")
print(df[['distance_km', 'delay_hours']].corr())

kpi = df.groupby('driver_id').agg({
    'delivery_id': 'count',
    'delay_hours': 'mean',
    'cost_per_km': 'mean'
}).rename(columns={'delivery_id': 'рейсов'})
print("=== KPI водителей ===")
print(kpi.round(2))

df.to_sql('delivery_analytics', engine, if_exists='replace')

=== Средняя задержка по погоде ===
weather_conditions
Clear     0.100
Cloudy    0.375
Fog       1.500
Rain      2.000
Snow      3.500
Name: delay_hours, dtype: float64
=== Средняя задержка по водителям ===
driver_id
1    0.200000
3    0.428571
4    0.625000
2    0.785714
5    2.285714
Name: delay_hours, dtype: float64
=== Корреляция расстояние и задежка ===
             distance_km  delay_hours
distance_km     1.000000    -0.352724
delay_hours    -0.352724     1.000000
=== KPI водителей ===
           рейсов  delay_hours  cost_per_km
driver_id                                  
1               5         0.20        11.73
2               7         0.79        12.30
3               7         0.43        13.26
4               4         0.62        11.75
5               7         2.29        13.59


30